# AIHub LowQualityPhoneVoice (저음질 전화망) — EDA

`/data/ASR/RAW/AIHub_LowQualityPhoneVoice/007.저음질_전화망_음성인식_데이터/01.데이터`

## 구조 (확인됨)
- `{1.Training, 2.Validation}/{라벨링데이터_230316, 원천데이터_230316}/`
- 라벨: `TL_D##`(train)·`VL_D##`(valid)`/D##/J##/S######/S######.json` (+발화별 txt는 JSON과 중복)
- 원천: `TS_D##`·`VS_D##/D##/J##/S######/NNNN.wav` — **8kHz mono PCM_16 확정**
- zip은 백업용(사용 안 함)

## 스키마 (확인됨) — 전사·duration이 JSON에 직접!
- `dataSet.typeInfo.{category, subcategory, inputType, speakers[].{id,type(상담사1/고객1),age,gender,residence,telephone_network}}`
- `dataSet.dialogs[].{speaker, audioPath(D##/J##/S######/NNNN.wav), duration, text}`
- → **발화별 txt를 안 읽고 JSON만 읽으면 됨** (파싱 매우 빠름)

## 규모
- train 188,870세션 / valid **1,926세션·39,916발화** ← 벤치마크 대상, 전수 스캔 가능

## 유의
- 전사에 `n/` 확인 → KsponSpeech식 컨벤션 (정규화 재사용)
- `telephone_network`(wide_band 등) — 저음질 특성 메타, 분포 확인
- 통화 데이터 → PII 가능성

In [1]:
from pathlib import Path
import re, json, random, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda env 확인 (TRAIN-ASR)")

DATA = Path("/data/ASR/RAW/AIHub_LowQualityPhoneVoice/007.저음질_전화망_음성인식_데이터/01.데이터")

def label_to_audio_base(tl_dir):
    """라벨 TL_D##/VL_D## 디렉터리 → 원천 TS_D##/VS_D## 디렉터리"""
    return Path(str(tl_dir).replace("라벨링데이터", "원천데이터")
                          .replace("TL_", "TS_").replace("VL_", "VS_"))

print("DATA 존재:", DATA.is_dir())
print("Training :", (DATA/"1.Training").is_dir(), "| Validation:", (DATA/"2.Validation").is_dir())

DATA 존재: True
Training : True | Validation: True


## 1. 매니페스트 파서 (JSON만 읽음 — txt 불필요)

> Validation은 1,926세션이라 **전수**, Training은 `TRAIN_SAMPLE` 세션만 표본. text·duration이 JSON에 있어 오디오/txt 안 열고도 분포·총시간 계산 가능.

In [2]:
TRAIN_SAMPLE = 300   # train 표본 세션 수 (전수는 188,870개라 EDA에선 표본만)

def build_manifest(label_root, split, max_sessions=None):
    rows = []
    jsons = sorted(label_root.rglob("S*.json"))
    if max_sessions:
        jsons = random.Random(0).sample(jsons, min(max_sessions, len(jsons)))
    for jp in jsons:
        try:
            ds = json.loads(jp.read_text(encoding="utf-8", errors="replace"))["dataSet"]
        except Exception:
            continue
        ti  = ds.get("typeInfo", {})
        spk = {s["id"]: s for s in ti.get("speakers", [])}
        # jp = .../TL_D##/D##/J##/S######/S######.json → parents[3] = TL_D##
        audio_base = label_to_audio_base(jp.parents[3])
        for d in ds.get("dialogs", []):
            sid = d.get("speaker"); sm = spk.get(sid, {})
            rows.append({
                "split": split, "domain": ti.get("category"), "subcat": ti.get("subcategory"),
                "session": jp.stem, "utt": Path(d.get("audioPath","")).stem,
                "speaker": sid, "spk_type": sm.get("type"),
                "gender": sm.get("gender"), "age": sm.get("age"),
                "residence": sm.get("residence"), "tel_net": sm.get("telephone_network"),
                "duration": d.get("duration"), "text": d.get("text",""),
                "audio_path": str(audio_base / d.get("audioPath","")),
            })
    df = pd.DataFrame(rows)
    if len(df):
        df["text"] = df["text"].astype("object")
    return df

df_v = build_manifest(DATA/"2.Validation/라벨링데이터_230316", "valid")             # 전수
df_t = build_manifest(DATA/"1.Training/라벨링데이터_230316", "train", TRAIN_SAMPLE)  # 표본
df = pd.concat([df_v, df_t], ignore_index=True)
print(f"valid 전수: 세션 {df_v['session'].nunique():,} / 발화 {len(df_v):,} / 총 {df_v['duration'].sum()/3600:.1f}h")
print(f"train 표본: 세션 {df_t['session'].nunique():,} / 발화 {len(df_t):,}  (TRAIN_SAMPLE={TRAIN_SAMPLE})")
df_v.head(3)

valid 전수: 세션 1,689 / 발화 39,916 / 총 57.1h
train 표본: 세션 299 / 발화 7,006  (TRAIN_SAMPLE=300)


,split,domain,subcat,session,utt,speaker,spk_type,gender,age,residence,tel_net,duration,text,audio_path
0,valid,교육,공부방법,S002538,0001,2,상담사1,남,20대,알수없음,8k,3.006,o/ 학습지원센터입니다. 무엇을 도와드릴까요?,/data/ASR/RAW/AIHub_LowQualityPhoneVoice/007.저...
1,valid,교육,공부방법,S002538,0002,1,고객1,여,50대,알수없음,8k,3.431,예 예 뭐 쯤 여쭤볼라 그러는데요.,/data/ASR/RAW/AIHub_LowQualityPhoneVoice/007.저...
2,valid,교육,공부방법,S002538,0003,1,고객1,여,50대,알수없음,8k,4.488,아무도 홈페이지에 아니 얘는 인강은 (1번)/(한 번)도 안 해봤어요.,/data/ASR/RAW/AIHub_LowQualityPhoneVoice/007.저...


In [3]:
# 매칭/결측 점검 (오디오 존재는 표본으로)
chk = df_v.sample(min(300, len(df_v)), random_state=0)
miss_audio = int(chk["audio_path"].map(lambda p: not Path(p).exists()).sum())
print(f"[valid] 오디오 존재 표본 {len(chk)} 중 없음: {miss_audio}")
print(f"[valid] 빈 전사: {int((df_v['text'].str.strip()=='').sum())} / duration 결측: {int(df_v['duration'].isna().sum())}")
print(f"[valid] 화자 메타 결측: gender {int(df_v['gender'].isna().sum())} / type {int(df_v['spk_type'].isna().sum())} / tel_net {int(df_v['tel_net'].isna().sum())}")

[valid] 오디오 존재 표본 300 중 없음: 0
[valid] 빈 전사: 0 / duration 결측: 0
[valid] 화자 메타 결측: gender 0 / type 0 / tel_net 0


## 2. 분포 (valid 전수 기준)

In [4]:
d = df_v
d["text_len"] = d["text"].str.len()
for col in ["domain", "subcat", "spk_type", "gender", "age", "residence", "tel_net"]:
    vc = d[col].value_counts(dropna=False)
    print(f"=== {col} ({d[col].nunique()}종) ===")
    print(vc.head(12).to_string())
    print()
print("=== 발화 duration(초) ===");  print(d["duration"].describe().round(2).to_string())
print("\n=== 전사 글자 수 ===");      print(d["text_len"].describe().round(1).to_string())
print(f"\nvalid 총 시간: {d['duration'].sum()/3600:.1f}h")

=== domain (4종) ===
domain
민원       15211
전자상거래    14105
교육        8664
HR        1936

=== subcat (20종) ===
subcat
결제, 취소, 환불 문의    6484
배송, 반송 문의        4334
강좌문의             4324
환경               2689
기타민원             2285
기타문의             2269
복지               2131
스마트기기            1958
문화및관광            1886
일반행정             1761
도시및경제            1524
교통및차량등록          1484

=== spk_type (6종) ===
spk_type
상담사1    22261
고객1     16843
상담사2      543
고객2       212
상담사3       55
상담사4        2

=== gender (2종) ===
gender
여    26418
남    13498

=== age (7종) ===
age
20대       13329
30대       12006
40대        6746
50대        4098
10대        2354
60대 이상     1374
None          9

=== residence (3종) ===
residence
알수없음    23326
경남      15211
서울경기     1379

=== tel_net (2종) ===
tel_net
8k           37980
wide_band     1936

=== 발화 duration(초) ===
count    39916.00
mean         5.15
std          3.09
min          1.50
25%          2.95
50%          4.21
75%          6.36
max         25.05

=== 전사 

## 3. 전사 컨벤션 (valid 전수)

> KsponSpeech식 태그·이중전사·@ 유무 전수 확인 → 정규화 규칙 재사용 여부 결정

In [5]:
txt = df_v["text"].fillna("")
print("특수문자 전수 (상위 25):")
print(txt.str.findall(r"[^가-힣a-zA-Z0-9\s]").explode().value_counts().head(25).to_string())
print(f"\n영문 포함: {txt.str.contains(r'[A-Za-z]').mean()*100:.2f}%  /  숫자 포함: {txt.str.contains(r'[0-9]').mean()*100:.2f}%")
print("\nKsponSpeech식 주석:")
for tag in ["b/", "n/", "l/", "o/", "u/", ")/(", ")("]:
    c = int(txt.str.contains(re.escape(tag)).sum())
    print(f"  '{tag}': {c:,}건")
print(f"  '@': {int(txt.str.contains('@').sum()):,}건  '+': {int(txt.str.contains(re.escape('+')).sum()):,}건  '*': {int(txt.str.contains(re.escape('*')).sum()):,}건")
print("\n전사 샘플 8개:")
for t in txt.sample(min(8, len(txt)), random_state=1):
    print("  •", t[:85])

특수문자 전수 (상위 25):
text
.    39252
/    28751
(    26604
)    26598
?    14186
,     5517
%      133
&       27
!        3
        2
ㅠ        1
ㄴ        1
>        1

영문 포함: 38.77%  /  숫자 포함: 11.88%

KsponSpeech식 주석:
  'b/': 1,383건
  'n/': 4,774건
  'l/': 157건
  'o/': 11,316건
  'u/': 0건
  ')/(': 6,966건
  ')(': 19건
  '@': 0건  '+': 0건  '*': 0건

전사 샘플 8개:
  • n/ 네, 알겠습니다.
  • o/ n/ 어 수능 대비 현자의 돌 모의고사 시즌 (1)/(원)이랑 (율) 환경 윤리론 맞으세요? 저희가 확인 후 안내 도와드리겠습니다. 잠시만 기다려주
  • n/ 아니요. 없습니다. 수고하세요.
  • 정보 확인하고 바로 안내 도와드릴 텐데요. 등록해 주셨을 때 회원가입 아 회원가입 해 주셨을 때 등록해 주신 번호가 어떻게 돼.
  • 수능완성 국어문학
  • 예. 신청을 아니 저기 (1번)/(한 번) 물어보라고요 올해 언제까지고 또 추후에 언제 받는다.
  • n/ 아 네 선생님 말씀은 이제 불법 쓰레기 투기는 아니고 뭐 종량제면 종량제 음식물 쓰레기
  • 안녕하세요. (NCS)/(엔씨에스) 교육 상담센터인가요?


## 3-1. 실제 음성 청취 — '저음질'이 어느 정도인지 직접 확인

In [6]:
for r in df_v.sample(3, random_state=1).itertuples():
    print(f"[{r.spk_type}/{r.gender}/{r.tel_net}] {r.text[:70]}")
    if sf and Path(r.audio_path).exists():
        data, sr = sf.read(r.audio_path)
        display(Audio(data, rate=sr))
    else:
        print("   (오디오 없음)")

[고객1/여/8k] n/ 네, 알겠습니다.


[상담사1/남/8k] o/ n/ 어 수능 대비 현자의 돌 모의고사 시즌 (1)/(원)이랑 (율) 환경 윤리론 맞으세요? 저희가 확인 후 안내 도와드


[고객1/여/wide_band] n/ 아니요. 없습니다. 수고하세요.


## 4. 오디오 속성 (valid 표본) — 8kHz 재확정 + JSON duration 대조

In [7]:
samp = df_v.sample(min(80, len(df_v)), random_state=0)
rows = []
for r in samp.itertuples():
    try:
        i = sf.info(r.audio_path)
        rows.append((i.samplerate, i.channels, i.subtype, round(i.frames/i.samplerate,2), r.duration))
    except Exception as e:
        rows.append(("ERR", str(e)[:20], "", None, r.duration))
a = pd.DataFrame(rows, columns=["sr","ch","subtype","dur_file","dur_json"])
print("sample_rate :", a["sr"].value_counts().to_dict())
print("channels    :", a["ch"].value_counts().to_dict())
print("subtype     :", a["subtype"].value_counts().to_dict())
ok = a.dropna(subset=["dur_file"])
ok = ok[ok["sr"] != "ERR"]
if len(ok):
    diff = (ok["dur_file"] - ok["dur_json"]).abs()
    print(f"JSON duration vs 파일 길이 차이: 평균 {diff.mean():.3f}s / 최대 {diff.max():.3f}s  (작으면 JSON duration 신뢰 가능)")

sample_rate : {8000: 80}
channels    : {1: 80}
subtype     : {'PCM_16': 80}
JSON duration vs 파일 길이 차이: 평균 0.002s / 최대 0.005s  (작으면 JSON duration 신뢰 가능)


## 5. 화자 분석 (valid)

In [8]:
print("화자 타입별 고유 화자 수(세션 내 id 기준):")
print(df_v.groupby("spk_type")["speaker"].nunique().to_string())
print("\n⚠ 이 데이터셋의 speaker id는 세션 내 '1','2'라 세션 간 동일인 추적 불가일 수 있음")
print("   → 화자 단위 분석은 (session, speaker) 조합 기준:")
df_v["spk_uid"] = df_v["session"] + "_" + df_v["speaker"].astype(str)
print(f"   (session,speaker) 조합 = {df_v['spk_uid'].nunique():,}개 / 세션 {df_v['session'].nunique():,}개")
print("\n조합별 발화 수: 평균 %.1f / 최대 %d" % (df_v["spk_uid"].value_counts().mean(), df_v["spk_uid"].value_counts().max()))

화자 타입별 고유 화자 수(세션 내 id 기준):
spk_type
고객1     3
고객2     2
상담사1    4
상담사2    4
상담사3    3
상담사4    1

⚠ 이 데이터셋의 speaker id는 세션 내 '1','2'라 세션 간 동일인 추적 불가일 수 있음
   → 화자 단위 분석은 (session, speaker) 조합 기준:
   (session,speaker) 조합 = 3,499개 / 세션 1,689개

조합별 발화 수: 평균 11.4 / 최대 236


In [9]:
# ============================================================
# 비식별화(PII) 토큰 포함 발화 → 전사 + 음성 직접 청취  (모든 EDA 노트북 공용)
# 파서로 DataFrame을 만든 셀을 먼저 실행한 뒤, 이 셀을 새 셀에 붙여 실행.
# DataFrame 변수(df/df_v/...)와 오디오 경로 컬럼(audio_path/wav/...)을 자동 탐지.
# ============================================================
import re
from pathlib import Path
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR")

import pandas as pd

# ---------- 설정 ----------
N_LISTEN = 5          # 들어볼 발화 수
RANDOM_STATE = 0      # None이면 매번 다른 표본
# 비식별화(PII) 토큰: 음향 토큰(b/ n/ l/ o/ u/)과 구분되는 익명화 전용 패턴
PII_PATTERNS = {
    "@ (이름 마커)":           re.compile(r"@"),
    "ㅇㅇ류 (익명화 2자+)":     re.compile(r"ㅇ{2,}"),
    "name/ (이름 태그)":        re.compile(r"(?:^|\s)name/", re.I),
    "[마스킹]":                re.compile(r"\[[^\]]{0,15}\]"),
    "<마스킹>":                re.compile(r"<[^>]{0,15}>"),
    "*** (별표 2+)":           re.compile(r"\*{2,}"),
    "xxx (엑스 2+)":           re.compile(r"[xX]{2,}"),
    "○○ (공백원 2+)":          re.compile(r"[○◯]{2,}"),
}

# ---------- 1) text DataFrame 자동 탐지 ----------
def _find_text_frame():
    g = globals()
    for name in ["df_v","df_valid","df","df_t","df_train","d","a"]:
        o = g.get(name)
        if isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    for name, o in g.items():
        if not name.startswith("_") and isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    return None, None

_name, _df = _find_text_frame()
if _df is None:
    raise RuntimeError("text 컬럼 DataFrame 없음 — 파서 셀을 먼저 실행하세요.")

# ---------- 2) 오디오 경로 컬럼 자동 탐지 ----------
AUDIO_COL = next((c for c in ["audio_path","wav","src_wav","audio","path","filepath"]
                  if c in _df.columns), None)
print(f"[대상] DataFrame '{_name}' · {len(_df):,} 발화 · 오디오 컬럼: {AUDIO_COL or '없음(PCM 직접 노트북일 수 있음)'}\n")

# ---------- 3) PII 토큰 집계 ----------
_txt = _df["text"].fillna("").astype("object")
print("=== 비식별화 토큰 집계 (text 원본) ===")
present = []
for label, pat in PII_PATTERNS.items():
    occ = int(_txt.str.count(pat).sum())
    utt = int(_txt.str.contains(pat).sum())
    if occ:
        present.append((label, pat, utt, occ))
        print(f"  {label:20s} 발화 {utt:>6,} · 출현 {occ:>6,} ({utt/len(_df)*100:.3f}%)")
if not present:
    print("  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.")

# ---------- 4) PII 포함 발화만 필터 → 전사 + 음성 재생 ----------
if present and AUDIO_COL:
    mask = pd.Series(False, index=_df.index)
    for _, pat, _, _ in present:
        mask |= _txt.str.contains(pat)
    hits = _df[mask]
    print(f"\n=== 비식별화 토큰 포함 발화 {len(hits):,}건 중 {min(N_LISTEN,len(hits))}개 청취 ===")
    print("   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)\n")
    sample = hits.sample(min(N_LISTEN, len(hits)), random_state=RANDOM_STATE)
    for r in sample.itertuples():
        text = getattr(r, "text", "")
        ap = getattr(r, AUDIO_COL, None)
        # 어떤 PII 패턴에 걸렸는지 표시
        tags = [lab for lab, pat, _, _ in present if pat.search(text or "")]
        print(f"[{', '.join(tags)}]")
        print(f"  전사: {text[:100]}")
        if sf and ap and Path(str(ap)).exists():
            try:
                data, sr = sf.read(str(ap))
                display(Audio(data, rate=sr))
            except Exception as e:
                print(f"   (재생 실패: {str(e)[:50]})")
        else:
            print(f"   (오디오 경로 없음/미존재: {ap})")
        print()
elif present and not AUDIO_COL:
    print("\n⚠ PII 토큰은 있으나 오디오 경로 컬럼을 못 찾음.")
    print("  이 노트북이 PCM을 직접 읽는 방식이면, 아래처럼 수동 지정:")
    print("  → sample = _df[mask].sample(N_LISTEN); 각 행의 키로 원본 PCM 경로를 구성해 재생")

[대상] DataFrame 'df_v' · 39,916 발화 · 오디오 컬럼: audio_path

=== 비식별화 토큰 집계 (text 원본) ===
  xxx (엑스 2+)          발화     25 · 출현     25 (0.063%)

=== 비식별화 토큰 포함 발화 25건 중 5개 청취 ===
   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)

[xxx (엑스 2+)]
  전사: o/ 메일을 보냈고 전화를 한 번 다시 달라고 ((xxx 전화드렸어요.))



[xxx (엑스 2+)]
  전사: o/네. 그 교재 교환 때문에 전화드렸는데 통화 괜찮으실까요? 아 네 다른 게 아니고요 교재 지금 필기한(((xx))) 반납하기 좀 불편하시다 하셔가지고 혹시 그러시면은 지금 어 



[xxx (엑스 2+)]
  전사: o/아 네 확인 감사합니다. 어 지금 구매하신 거 어 (()) ((xxx취합)) 후 환불 다 가능하신데요 이거 사유는 어떻게 되세요?



[xxx (엑스 2+)]
  전사: 아 그 ((x수 학생xx)) 뉴스타트 패키지로 다시 할려고



[xxx (엑스 2+)]
  전사: 아니요. 어머니 ((xxx)) 구매하실 때
